# Ch7 (A, inline) - Tuning the Encoders: Embedding SFT

v-space pulls a question to the right attribute; u-space separates a true claim from a conflicting-value one. Both tune a low-rank adapter over a frozen base, on the Ch6 generated data.

In [ ]:
# knowlytix and forgeloop are installed from PyPI (pip install knowlytix forgeloop)
import os, sys
REPO = os.path.join(os.path.dirname(os.getcwd()), "code") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(REPO, "scripts"))

In [ ]:
from knowlytix.embedding import EmbeddingSFTConfig, finetune_embedding
import torch
import capstone_pipeline as cp
store = cp.load_store(os.path.join(REPO, "data", "gms_annual_report_store"))
d_v = store.model.cfg.d_v
v_ft = finetune_embedding(os.path.join(REPO,"data","enrichment","embedding_sft.jsonl"),
    EmbeddingSFTConfig(rank=8, mode="full", out_dim=d_v, objective="prototype"),
    text_col="text", label_col="label")
print("v-space val_accuracy =", round(v_ft.val_accuracy, 3))

u-space: genuine conflicting-value pairs, full mode (a rotation cannot move tension).

In [ ]:
from finetune_encoders import _contradiction_pairs, _fit_contradiction_pairs, _uspace_tension
pos, neg = _contradiction_pairs(store)
u_ft = _fit_contradiction_pairs(pos, neg, EmbeddingSFTConfig(
    rank=32, mode="full", objective="contradiction",
    encoder="sentence-transformers/nli-mpnet-base-v2", out_dim=d_v, epochs=400))
print("consistent:", round(_uspace_tension(u_ft, "Retail\u2019s headcount was 520.",
      "The headcount of Retail is 520."), 3))
print("contradictory:", round(_uspace_tension(u_ft, "Retail\u2019s headcount was 520.",
      "Retail\u2019s headcount was 210."), 3))